In [1]:
%cd /kaggle/working

!git clone https://github.com/BillChan226/AgentPoison.git

%cd AgentPoison

!ls

/kaggle/working
fatal: destination path 'AgentPoison' already exists and is not an empty directory.
/kaggle/working/AgentPoison
agentdriver  assets    embedder		LICENSE  README.md  scripts
algo	     EhrAgent  environment.yml	ReAct	 results    wandb


In [2]:
!pip install jsonlines
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.3 MB/s eta 0:00:00:00:0100:01


In [3]:
!wandb offline

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
# ### 请确保当前路径在 AgentPoison 根目录下
# # 记得先创建 results 文件夹: mkdir -p ./results

# !python algo/trigger_optimization.py \
#   --agent qa \
#   --algo ap \
#   --model dpr-ctx_encoder-single-nq-base \
#   --save_dir ./results \
#   --num_iter 1000 \
#   --num_grad_iter 30 \
#   --num_cand 100 \
#   --num_adv_passage_tokens 5 \
#   --target_gradient_guidance \
#   --ppl_filter \
#   --asr_threshold 0.5 \
#   --report_to_wandb 


# # !python algo/trigger_optimization.py --agent qa --algo ap --model dpr-ctx_encoder-single-nq-base --save_dir ./results  --ppl_filter --target_gradient_guidance --asr_threshold 0.5 --num_adv_passage_tokens 10 --golden_trigger -w -p

In [13]:
class GradientStorage:
    """
    This object stores the intermediate gradients of the output a the given PyTorch module.
    """
    def __init__(self, module, num_adv_passage_tokens):
        self._stored_gradient = None
        self.num_adv_passage_tokens = num_adv_passage_tokens
        module.register_full_backward_hook(self.hook)
        self.call_count = 0

    def hook(self, module, grad_in, grad_out):
        self.call_count += 1
        if self._stored_gradient is None:
            self._stored_gradient = grad_out[0][:, -self.num_adv_passage_tokens:]
        else:
            self._stored_gradient += grad_out[0][:, -self.num_adv_passage_tokens:]

    def get(self):
        return self._stored_gradient

    # 【新增这个方法】用来手动清空梯度
    def zero_grad(self):
        # return
        self._stored_gradient = None

    def get_call_count(self):
        # following the original paper, this should increase (BUG)
        return self.call_count

In [6]:
# --- 0. 导入必要的库 ---
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
import numpy as np
import os, sys, random, gc, datetime, pickle, json
from tqdm.notebook import tqdm  # 使用 notebook 专用的进度条
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

# ⚠️ 确保你已经在上一个 Cell 定义了 utils 中的函数
# 如果你没有运行上一个 Cell，请取消下面这行的注释并指向正确路径
sys.path.append("/kaggle/working/AgentPoison")
from algo.utils import *
# --- 1. 定义配置参数 (替代 argparse) ---
from algo.trigger_optimization import (
    # GradientStorage,
    compute_avg_cluster_distance,
    hotflip_attack,
    candidate_filter,
    # expanded_cluster_centers,
    # compute_avg_embedding_similarity
)


class Args:
    # 核心任务配置
    agent = "qa"           # 任务: qa (StrategyQA), ad (AgentDriver)
    algo = "ap"            # 算法: ap (AgentPoison)
    
    # 模型配置 (请确保这些 Key 在你的 utils.py 字典里有定义)
    model = "dpr-ctx_encoder-single-nq-base" 
    
    # 路径配置
    save_dir = "./results"
    
    # 优化参数
    num_iter = 20 # 1000        # 迭代次数 (测试时可以设小一点，比如 100)
    num_grad_iter = 10 # previously 1, lead to poor batch size      # 梯度累积步数 (显存不够就调大这个)
    per_gpu_eval_batch_size = 32  # Batch Size (显存爆了就调小: 16 -> 8 -> 4)
    num_cand = 100          # 每次采样的候选词数量
    num_adv_passage_tokens = 5    # Trigger 长度 (StrategyQA 论文用的是 5)
    
    # 高级开关
    golden_trigger = True # 是否从特定 Trigger 开始
    target_gradient_guidance = False # 是否开启 LLM 引导 (显存不够建议 False)
    use_gpt = False        # 是否用 GPT-3.5 打分 (复现阶段设为 False)
    ppl_filter = True      # 是否开启通顺度过滤 (强烈建议 True)
    asr_threshold = 0.5
    
    # 调试
    plot = False
    report_to_wandb = False # 彻底关闭 WandB

args = Args()
print("environment imported")

2026-01-03 14:35:34.130011: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767450934.322931      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767450934.377094      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767450934.837521      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767450934.837559      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767450934.837562      55 computation_placer.cc:177] computation placer alr

environment imported


In [7]:
load_db_qa

<function algo.utils.load_db_qa(database_samples_dir='ReAct/database/strategyqa_train_paragraphs.json', db_dir='data/memory', model_code='None', model=None, tokenizer=None, device='cuda')>

In [8]:
# --- 2. 环境初始化 ---
# 自动检测设备，防止报错
device = "cuda:0" if torch.cuda.is_available() else "cpu"
target_device = device
print(f"🚀 Running on device: {device}")

# 创建保存目录
root_dir = f"{args.save_dir}/{args.agent}/{args.algo}/{str(datetime.datetime.now()).replace(' ', '_')}"
os.makedirs(root_dir, exist_ok=True)
print(f"📂 Results will be saved to: {root_dir}")

🚀 Running on device: cuda:0
📂 Results will be saved to: ./results/qa/ap/2026-01-03_14:35:53.029946


In [9]:
!mkdir ReAct/database/embeddings

mkdir: cannot create directory ‘ReAct/database/embeddings’: File exists


In [10]:
# adapted from agent poison repository, to control the "device" parameter
def compute_perplexity(input_ids, model, device=device):
    """
    Calculate the perplexity of the input_ids using the model.
    """

    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    loss, logits = outputs[:2]
    return torch.exp(loss)


In [11]:
# adapted from agent poison repository, to control the "device" parameter
import random # probability output
def candidate_filter(candidates,
            num_candidates=1,
            token_to_flip=None,
            adv_passage_ids=None,
            ppl_model=None, 
            ppl_tokenizer=None,
            model_tokenizer=None,
            device=device):
    """Returns the top candidate with max ppl."""
    with torch.no_grad():
    
        ppl_scores = []
        temp_adv_passage = adv_passage_ids.clone()
        for candidate in candidates:
            temp_adv_passage[:, token_to_flip] = candidate
            # 1. BERT ID -> 文本 # TODO now only 1 trigger, so ok to take [0]
            text = model_tokenizer.batch_decode(temp_adv_passage, skip_special_tokens=False) #True)
            text[0] = text[0].replace("##", "")
            # sample = random.random()
            sample = 1 # do not print
            if sample < 0.05:
                print(text)
            # 2. 文本 -> GPT-2 ID
            gpt2_inputs = ppl_tokenizer(text, return_tensors="pt").to(device)

            if sample < 0.05:
                print(gpt2_inputs)
                first_sample_ids = gpt2_inputs.input_ids[0]
                # 把 ID 变回 Token 字符串列表
                gpt2_tokens = ppl_tokenizer.convert_ids_to_tokens(first_sample_ids)
                print(gpt2_tokens)
            # 3. GPT-2 ID -> 模型
            ppl_score = compute_perplexity(gpt2_inputs.input_ids, ppl_model, device) * -1

            
            # ppl_score = compute_perplexity(temp_adv_passage, ppl_model, device) * -1
            ppl_scores.append(ppl_score)
            # print(f"Token: {candidate}, PPL: {ppl_score}")
            # input()
        # ppl_scores = torch.tensor(ppl_scores)
        ppl_scores = torch.stack(ppl_scores)
        _, top_k_ids = ppl_scores.topk(num_candidates)
        candidates = candidates[top_k_ids]

    return candidates


In [14]:
# --- 3. 核心逻辑 (从源码 main 块提取) ---
from transformers import AutoModelForCausalLM, AutoTokenizer

# A. 加载模型 (Retriever)
print("⏳ Loading Embedder Model...")
# e.g. dpr
model, tokenizer, get_emb = load_models(args.model, device)
model.eval() # 冻结模型

# B. 初始化 Trigger
print("🔧 Initializing Trigger...")
if not args.golden_trigger:
    # 默认从 [MASK] 开始
    adv_passage_ids = [tokenizer.mask_token_id] * args.num_adv_passage_tokens
    adv_passage_ids = torch.tensor(adv_passage_ids, device=device).unsqueeze(0)
else:
    # 如果你想从一句人话开始优化
    initial_trigger = "Make efficient calls."
    adv_passage_ids = tokenizer(initial_trigger, return_tensors="pt", truncation=True, max_length=args.num_adv_passage_tokens).input_ids.to(device)

print(f"📝 Initial Trigger Tokens: {tokenizer.convert_ids_to_tokens(adv_passage_ids[0])}")

# 获取 Embedder 的词向量层 (用于计算梯度)
embeddings = get_embeddings(model)
embedding_gradient = GradientStorage(embeddings, args.num_adv_passage_tokens)

# C. 加载辅助模型 (PPL Filter & Target Guidance)



# 1. 定义映射字典
model_code_to_embedder_name = {
    # 选项 A: TinyLlama (推荐调试用，无需权限，下载快，显存小)
    "tiny-llama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    
    # 选项 B: Llama-2-7b (官方版，需要 HuggingFace Token 和 meta-llama 仓库权限)
    "meta-llama-2-chat-7b": "meta-llama/Llama-2-7b-chat-hf",
    
    # 选项 C: Llama-3-8b (最新版)
    "meta-llama-3-8b-instruct": "meta-llama/Meta-Llama-3-8B-Instruct"
}
ppl_model = None
if args.ppl_filter:
    print("⏳ Loading GPT-2 for Perplexity Filter...")
    # 确保你的 utils.py 里 gpt2 的路径是对的，或者直接用 "gpt2"
    ppl_model, ppl_tokenizer, _ = load_models("gpt2", device)
    # ppl_model, ppl_tokenizer, _ = load_models("llama2", device)

    # 2. 设置你当前想用的 model_code
    # 建议先用 tiny-llama 跑通流程
    # model_code = "tiny-llama" 
    # model_code = "meta-llama-2-chat-7b" 
    
    # 3. 你的原始代码 (稍微调整了格式以确保运行)
    #######################################
    # print(f"⏳ Loading PPL Model: {model_code_to_embedder_name[model_code]} ...")
    
    # ppl_model = AutoModelForCausalLM.from_pretrained(
        # model_code_to_embedder_name[model_code], 
        # load_in_8bit=True,   # 需要安装 bitsandbytes
        # device_map={"": device}
    # )
    
    # ppl_tokenizer = AutoTokenizer.from_pretrained(model_code_to_embedder_name[model_code])
    
    # 确保 Tokenizer 有 pad_token (Llama 默认没有，为了 batch 处理必须加)
    # if ppl_tokenizer.pad_token is None:
        # ppl_tokenizer.pad_token = ppl_tokenizer.eos_token
    ################################################

    ppl_model.eval()

target_model = None
target_gradient_guidance = args.target_gradient_guidance
if target_gradient_guidance:
    print("⏳ Loading Target LLM for Guidance...")
    # 这里记得用 TinyLlama 替代 Llama2
    target_model_code = "meta-llama-2-chat-7b" 
    target_model, target_tokenizer, get_target_emb = load_models(target_model_code, device)
    target_model.eval()
    target_model_embeddings = get_embeddings(target_model)
    target_embedding_gradient = GradientStorage(target_model_embeddings, args.num_adv_passage_tokens)

# D. 准备数据 (Loading Dataset)
print("📚 Loading StrategyQA Dataset...")
# 注意：这里需要确保你已经把数据上传到了指定位置，或者修改这里的路径
if args.agent == "qa":
    # 假设你还没上传，为了跑通代码，我们先伪造一个小数据集
    # 如果你上传了数据，请取消下面注释并修改路径
    # database_samples_dir = "ReAct/database/strategyqa_train_paragraphs.json"
    # test_samples_dir = "ReAct/database/strategyqa_train.json"
    # db_embeddings = load_db_qa(...) 
    database_samples_dir = "ReAct/database/strategyqa_train_paragraphs.json"
    test_samples_dir = "ReAct/database/strategyqa_train.json"
    # test_samples_dir = "ReAct/exp_6_15/intermediate.json"
    db_dir = "ReAct/database/embeddings"
    # Load the database embeddings
    model_code = args.model
    db_embeddings = load_db_qa(database_samples_dir, db_dir, model_code, model, tokenizer, device)

    split_ratio = 1.0
    train_dataset = StrategyQADataset(test_samples_dir, split_ratio=split_ratio, train=True)
    valid_dataset = StrategyQADataset(test_samples_dir, split_ratio=split_ratio, train=False)
    # slice = 998 # from index 998, the tokens have meaning, ie not [PAD], [SEP], [MASK]...
    slice = 0

train_dataloader = DataLoader(train_dataset, batch_size=args.per_gpu_eval_batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=args.per_gpu_eval_batch_size, shuffle=False)


# cluster the dataset to avoid calculating every embedding, but only calculate the centroid
# why : 1. reduce computation, 2. prevent overfitting.
# if no clustering, the result could be between two very close embedding points, but the "global distance average" is still minimized
gmm = GaussianMixture(n_components=5, covariance_type='full', random_state=0)
gmm.fit(db_embeddings.cpu().detach().numpy())
cluster_centers = gmm.means_
cluster_centers = torch.tensor(cluster_centers).to(device)
expanded_cluster_centers = cluster_centers.unsqueeze(0)


# load all data into memory, not needed in replication
# elif args.agent == "qa":
#     query_samples = []
#     all_data = {"question":[]}
#     for idx, batch in enumerate(train_dataloader):
#         question_batch = batch["question"]
#         for question in question_batch:
#             query_samples.append(question)
#             all_data["question"].append(question)


# E. 核心优化循环 (Algorithm 1)
print("🔥 Starting Optimization Loop...")
adv_passage_attention = torch.ones_like(adv_passage_ids, device=device)

# 进度条
pbar = tqdm(range(args.num_iter), desc="Optimizing")

train_iter = iter(train_dataloader)

for it_ in pbar:
    # 1. 梯度计算 (Gradient Accumulation)
    model.zero_grad()
    loss_sum = 0
    grad = None

    current_loss_avg = 0  # 用来记录这一轮的平均 Loss，作为后续比较的基准

    embedding_gradient.zero_grad() # the original code did not empty gradient
    
    for _ in range(args.num_grad_iter):
        ###
        try:
            data = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dataloader)
            data = next(train_iter)

        # 前向传播计算 Loss
        if args.agent == "qa":
            query_embeddings = bert_get_adv_emb(data, model, tokenizer, args.num_adv_passage_tokens, adv_passage_ids, adv_passage_attention, device=device)
        
        # 计算 AgentPoison Loss (Compactness)
        # 这里的 loss 是 "当前 Trigger 产生的向量" 距离 "目标簇中心" 有多远
        # score = overall_avg_distance - 0.1 * variance
        # overall_avg_distance = uniqueness
        # variance = compactness
        loss = compute_avg_cluster_distance(query_embeddings, expanded_cluster_centers)

        # 累积 Loss (用于显示和比较)
        current_loss_avg += loss.item()
        
        # loss_sum = loss.item()
        loss.backward() # 反向传播获取梯度

        # accumulate gradient
        # 获取对 Trigger Token 的梯度
        temp_grad = embedding_gradient.get()
        # grad = temp_grad.sum(dim=0) # 简化处理，直接用 Sum

        if grad is None:
            grad = temp_grad.sum(dim=0) / args.num_grad_iter
        else:
            grad += temp_grad.sum(dim=0) / args.num_grad_iter
    
    current_loss_avg /= args.num_grad_iter
    print("call count",embedding_gradient.get_call_count())
    # 2. HotFlip 攻击 (寻找候选词)
    token_to_flip = random.randrange(args.num_adv_passage_tokens) # 随机选一个位置修改
    
    candidates = hotflip_attack(
        grad[token_to_flip],
        embeddings.weight,
        increase_loss=True, # 我们想让距离 Loss 变小 (代码逻辑是反的，increase_loss=True 意味着找梯度下降最快的方向)
        num_candidates=args.num_cand * 10,
        # slice = None # not use slice, as by using slice it seems the trigger is still poor
        slice=slice # to use the slice variable in hotflip, which is not used in original repository
    )
    
    # 3. 过滤候选词 (PPL Filter)
    if args.ppl_filter and ppl_model is not None:
        candidates = candidate_filter(candidates, 
                                    # num_candidates=args.num_cand // 2, # 过滤掉一半
                                    num_candidates=args.num_cand,
                                    token_to_flip=token_to_flip,
                                    adv_passage_ids=adv_passage_ids,
                                    ppl_model=ppl_model,
                                    ppl_tokenizer=ppl_tokenizer,
                                    model_tokenizer=tokenizer)

    # 4. 评估候选词 (Evaluation)
    best_candidate_idx = 0
    best_candidate_score = -float("inf")
    
    for i, candidate in enumerate(candidates):
        # A. 构建临时 Trigger
        temp_adv_passage = adv_passage_ids.clone()
        temp_adv_passage[:, token_to_flip] = candidate
        
        # B. 重新计算 Embedder Loss (Compactness/Uniqueness)
        # 这一步是必须的，因为 HotFlip 只是近似，真实 Loss 可能会不同
        with torch.no_grad():
            if args.agent == "qa":
                cand_embeddings = bert_get_adv_emb(data, model, tokenizer, args.num_adv_passage_tokens, temp_adv_passage, adv_passage_attention, device=device)
                cand_loss = compute_avg_cluster_distance(cand_embeddings, expanded_cluster_centers)
                cand_score = cand_loss.item()

        # C. [关键] LLM Target Guidance (诱导生成概率)
        # 如果开启了引导，这里会覆盖上面的 Score
        if args.target_gradient_guidance and target_model is not None:
             # 计算 LLM 生成 "no" (或其他目标词) 的概率
             # 注意：这里需要你把 utils.py 里的 "STOP" 硬编码改成 args.target_token
             guidance_score = target_word_prob(
                 data, 
                 target_model, 
                 target_tokenizer, 
                 args.num_adv_passage_tokens, 
                 temp_adv_passage, 
                 adv_passage_attention, 
                 "no", # <--- 这里填入你的目标词，StrategyQA 建议用 "no"
                 "",   # CoT Prefix (QA任务通常为空)
                 "",   # Trigger Sequence (函数内部会重新拼装)
                 target_device
             )
             
             # 如果开启引导，我们主要看 Guidance Score，但通常也希望 Embedder Loss 别太差
             # 简单的策略：直接用 Guidance Score 作为评判标准
             final_score = guidance_score
        else:
             # 如果没开启引导，就只看 Embedder 的 Score
             final_score = cand_score

        # D. 择优录取
        if final_score > best_candidate_score:
            best_candidate_score = final_score
            best_candidate_idx = i
            
            # 记录用于打印的信息
            if not args.target_gradient_guidance:
                 # 如果是纯 AP 算法，我们关注 Loss (Score)
                 current_best_loss = cand_score

    # E. 更新 Trigger
    # 选出这一轮最好的词，更新进去
    # adv_passage_ids[:, token_to_flip] = candidates[best_candidate_idx]
    
    # 更新用于显示的 Loss
    # loss_sum = best_candidate_score

    # --- E. 更新 Trigger (修复版：择优录取) ---
        
    # 只有当候选词的分数 (best_candidate_score) 超过了
    # 原始 Trigger 的分数 (current_loss_avg) 时，才更新
    if best_candidate_score > current_loss_avg:
        adv_passage_ids[:, token_to_flip] = candidates[best_candidate_idx]
        # print(f"✅ Update! Gain: {best_candidate_score - current_loss_avg:.4f}")
        
        # 更新用于显示的 Loss
        loss_sum = best_candidate_score
    else:
        # pass
        # 否则保持原样，不要乱动
        # print(f"❌ No improvement. (Best: {best_candidate_score:.4f} vs Current: {current_loss_avg:.4f})")
        loss_sum = current_loss_avg

    
    # 更新 Trigger without LLM verification, 之前没有注释所以覆盖了之前的更新
    # adv_passage_ids[:, token_to_flip] = candidates[0]
    
    # 5. 打印进度
    current_trigger_tokens = tokenizer.convert_ids_to_tokens(adv_passage_ids[0])
    current_trigger_str = tokenizer.decode(adv_passage_ids[0])
    
    # 更新进度条信息
    pbar.set_postfix({'Loss': f"{loss_sum:.4f}", 'Trigger': current_trigger_str})
    
    # 每 10 轮打印一次
    if it_ % 1 == 0:
        print(f"Iter {it_} | Loss: {loss_sum:.4f} | Trigger: {current_trigger_tokens}")

# --- 4. 结果输出 ---
final_trigger = tokenizer.decode(adv_passage_ids[0])
print("\n" + "="*30)
print(f"🎉 Optimization Finished!")
print(f"☠️  Final Trigger: {final_trigger}")
print("="*30)

# 把结果保存到文件
with open(f"{root_dir}/final_trigger.txt", "w") as f:
    f.write(final_trigger)

⏳ Loading Embedder Model...


Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


🔧 Initializing Trigger...
📝 Initial Trigger Tokens: ['[CLS]', 'make', 'efficient', 'calls', '[SEP]']
⏳ Loading GPT-2 for Perplexity Filter...
📚 Loading StrategyQA Dataset...
🔥 Starting Optimization Loop...


Optimizing:   0%|          | 0/20 [00:00<?, ?it/s]sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


call count 320


...

Iter 2 | Loss: 13.6284 | Trigger: ['[MASK]', 'make', 'vane', 'braced', '[SEP]']
call count 1280


Optimizing:  20%|██        | 4/20 [10:08<40:29, 151.83s/it, Loss=13.5927, Trigger=[MASK] make vane braced [SEP]]

Iter 3 | Loss: 13.5927 | Trigger: ['[MASK]', 'make', 'vane', 'braced', '[SEP]']


Optimizing:  20%|██        | 4/20 [10:14<40:58, 153.67s/it, Loss=13.5927, Trigger=[MASK] make vane braced [SEP]]


KeyboardInterrupt: 

### AgentPoison Reproduction Experiment Record (ReAct-StrategyQA)

**Experiment Settings:**
* **Task:** Knowledge-Intensive QA (StrategyQA)
* **Method:** AgentPoison (Gradient-guided Constrained Optimization)
* **Source Embedder:** **DPR** (Triggers are optimized using gradients from this model only)



| Target Embedder | ASR-r (Retrieval Success Rate) | ASR-a (Action Success Rate) | ASR-t (End-to-End Success Rate) | ACC (Benign Accuracy) |
| :--- | :--- | :--- | :--- | :--- |
| **DPR** | [cite_start]*(Target: >70%)* [cite: 547-558, 15] | [cite_start]*(Target: >72%)* [cite: 547-558, 15] | [cite_start]*(Target: >59%)* [cite: 547-558, 15] | [cite_start]*(Target: >90%)* [cite: 547-558, 15] |
| **ANCE** | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **BGE** | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **REALM** | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **ORQA** | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **OpenAI Ada-002** | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |

### AgentPoison Reproduction Experiment Record (ReAct-StrategyQA)

**Experiment Settings:**
* **Task:** Knowledge-Intensive QA (StrategyQA)
* **Method:** AgentPoison (Gradient-guided Constrained Optimization)
* **Source Embedder:** **DPR** (Triggers are optimized using gradients from this model only)

| Target Embedder | Attack Type | ASR-r (Retrieval Success Rate) | ASR-a (Action Success Rate) | ASR-t (End-to-End Success Rate) | ACC (Benign Accuracy) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **DPR** | **White-box Attack (Self)** | *(Target: >70%)* | *(Target: >72%)* | *(Target: >59%)* | *(Target: >90%)* |
| **ANCE** | Transfer Attack | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **BGE** | Transfer Attack | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **REALM** | Transfer Attack | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **ORQA** | Transfer Attack | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |
| **OpenAI Ada-002** | Transfer Attack | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* | *(Actual Value)* |